In [ ]:
import os
import pyreadr
import json
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from notebook_utils import NOTEBOOK_DIR
from cityscaper.utils import read_rds_to_df, resolve_path, sorted_columns, latlon_filter, geojson_rds_to_json, geojson_to_parcel_bounds

DATA_DIR = NOTEBOOK_DIR.parent / "data"
OUTPUT_DIR = DATA_DIR / "output"

%matplotlib inline

In [ ]:
# Load reference data for exploration
raw_geom_geojson = geojson_rds_to_json(DATA_DIR / "sf_map.RDS")
clean_geom = geojson_to_parcel_bounds(raw_geom_geojson)
geom_series = pd.Series(clean_geom)

fname = DATA_DIR / "five_rezonings_nongeo.RDS"
rezoning_base = read_rds_to_df(DATA_DIR / "five_rezonings_nongeo_unfiltered.RDS", index_cols='mapblklot')
print('\n'.join(sorted_columns(rezoning_base)))
rezoning_base.loc['3561025',['lat', 'lng', 'year', 'pdev_baseline_1yr', 'ex_height2024']]

In [ ]:
geom_select = [-122.43270,37.76874,-122.43060,37.77047]
rezoning_scenario = "F"  # Family rezoning

rezoning_fname = f"rezoning_{rezoning_scenario}_output.RDS"
rezoning_scenario_data = read_rds_to_df(DATA_DIR / rezoning_fname, index_cols='mapblklot')
assert 'height' in rezoning_scenario_data.columns
assert 'pdev' in rezoning_scenario_data.columns

lots_in_region = latlon_filter(rezoning_scenario_data, *geom_select)
development_candidates = lots_in_region[lots_in_region['ZONING'].notnull()].copy()

modeling_variables = [c for c in development_candidates.columns if 'pdev' in c] + \
                     ['M4_ZONING', 'M5_ZONING', 'M6_ZONING', 'height', 'ZONING','M6_height',
                      'height', 'lot_coverage_discount', 'ground_floor', 'ACRES', 'ex_height2024']
print("\n".join(modeling_variables))
# rezoning_scenario_data.loc['3561025',['M4_ZONING', 'M5_ZONING', 'M6_ZONING', 'height', 'ZONING','M6_height']]

development_candidates[modeling_variables]

In [ ]:
import os
import csv
import json

header = """<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2" 
     xmlns:gx="http://www.google.com/kml/ext/2.2">
  <Document>
"""

template = """    <Placemark>
      <name>{mapblklot}</name>
      <description>{height}</description>
      <Polygon>
        <extrude>1</extrude>
        <altitudeMode>relativeToGround</altitudeMode>
        <outerBoundaryIs>
          <LinearRing>
            <coordinates>
{coordinate_string}
            </coordinates>
          </LinearRing>
        </outerBoundaryIs>
      </Polygon>
      <Style>
        <PolyStyle>
          <color>cc0000ff</color> <!-- Red with 80% opacity -->
        </PolyStyle>
      </Style>
    </Placemark>"""

footer="""  </Document>
</kml>"""

geometry_file = os.path.expanduser("~/src/cityscaper/data/sf_map_unfiltered.json")
csv_path = os.path.expanduser("~/Desktop/rezoning_output.csv")
csv_path = os.path.expanduser("~/Desktop/rezoning_geary.csv")
with open(geometry_file, "r") as f:
    geom_data = json.load(f)

with open(csv_path, newline="") as f:
    parcel_specs = list(csv.DictReader(f))

buildings = []
for i, row in enumerate(parcel_specs):
    lot = row["mapblklot"]
    try:
        height = float(row["height"])
        parcel_bounds = geom_data[lot]
        for j, polygon in enumerate(parcel_bounds):
            building_name = f"{lot}_{j+1}"
            vertex_strings = []
            for lat, lng in polygon:
                vertex_strings.append(f"              {lat},{lng},{height*0.3048}")
            coordinate_string="\n".join(vertex_strings)
            building_string = template.format(mapblklot=lot,
                                              coordinate_string=coordinate_string,
                                              height=height
                                             )
            buildings.append(building_string)
    except KeyError as e:
        continue

kml = header + "\n".join(buildings) + footer

print(kml)